In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
from numpy.ma import mean
from pathlib import Path

from cxd.core.spec import get_positions, Spec, SpecScan
from qorageom.io import load_mat

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

run: int = 946
spec_file: Path = Path(r"D:\05_Resources\Data\9C_260324\260324")
mat_file: Path = Path(r"D:\05_Resources\Data\9C_260324\analysis\ROI") / f"scan_{run:04d}.mat"

spec = Spec(spec_file)
scan: SpecScan = spec.get_scan(run)
df: pd.DataFrame = scan.to_dataframe()

pos_df: pd.DataFrame = get_positions(run_ns=[run], spec_file=spec_file)
frames: np.ndarray = load_mat(mat_file)
thetas: np.ndarray = df['Theta'].to_numpy()
mean_theta = mean(thetas)
print(f"Mean Theta: {mean_theta:.4f}°")
print(f"Theta shape: {thetas.shape}")
print(f"Frames shape: {frames.shape}")

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.imshow(np.log1p(frames.sum(axis=0)), origin='lower')
ax.set_title(f"Run {run} - Sum over Omega")

In [ ]:
pos_df.loc[run, 'bc']

In [ ]:
from cxd.core.transform import correct_rocking
from cxd.core.constants import TIMEPIX_PIXEL_SIZE

beam_center = (len(frames) // 2, pos_df.loc[run, 'bc']['y'], pos_df.loc[run, 'bc']['x'])
beam_center = np.array(beam_center)
corrected_frames = correct_rocking(
    frames,
    np.array([60, 125, 125]),
    thetas,
    scan.sdd,
    beam_energy=5.55,
    pixel_size=TIMEPIX_PIXEL_SIZE,
)

print(corrected_frames.shape)

In [ ]:
import pyvista as pv
pv.set_jupyter_backend('trame')
from cxd.core.visualize import visualize_q_space_vtk

visualize_q_space_vtk(corrected_frames, 0.01, True)
